In [17]:
import pandas as pd
import networkx as nx
import osmnx as ox
import numpy as np
import random

# 1. LOAD DATA
# Ensure these files are in your working directory
GRAPH_PATH = '/Users/ruhankarthik/SafePath/data/processed/sd_walk_graph.graphml'
SCORES_PATH = '/Users/ruhankarthik/SafePath/data/processed/edge_scores.csv'

print("Loading GraphML file... (This may take a minute for large files)")
# Using ox.load_graphml is the standard way to load the saved street network
G = ox.load_graphml(GRAPH_PATH)

print("Loading Scores CSV...")
df_scores = pd.read_csv(SCORES_PATH)

# Set (u, v, key) as index for high-speed mapping
df_scores.set_index(['u', 'v', 'key'], inplace=True)

# 2. DEFINE ROUTING PROFILES
# We minimize (1 - Score). 
# Higher score = Lower cost = More likely to be chosen.
profiles = {
    'Safety_Day': {
        'crime_score_day': 0.7, 
        'walk_score': 0.1, 
        'road_class_score': 0.2
    },
    'Safety_Night': {
        'crime_score_night': 0.5, 
        'light_score': 0.4, 
        'road_class_score': 0.1
    },
    'Standard_Shortest': {
        'walk_score': 0.1, 
        'road_class_score': 0.1  # Mostly relies on edge length
    }
}

def calculate_composite_cost(edge_data, weights):
    """Calculates weight-adjusted cost for an edge."""
    score = 0
    total_weight = sum(weights.values())
    
    # Map the internal graph data (from CSV) to weight keys
    # CSV Column names vs logic keys
    mapping = {
        'crime_score_day': 'crime_day',
        'crime_score_night': 'crime_night',
        'light_score': 'light',
        'walk_score': 'walk',
        'road_class_score': 'road'
    }
    
    for score_key, weight in weights.items():
        # Default to 0.5 (neutral) if data is missing for that edge
        val = edge_data.get(mapping[score_key], 0.5)
        score += val * weight
        
    # Cost = (1 - Normalized Score) * Length
    # This ensures we don't take a 10-mile detour just to avoid one 'un-lit' block
    return (1.0 - (score / total_weight)) * edge_data.get('length', 1.0)

# 3. ATTACH SCORES TO GRAPH
print("Merging scores into graph edges and calculating costs...")
for u, v, k, data in G.edges(keys=True, data=True):
    try:
        # Match scores from CSV to graph edge
        row = df_scores.loc[(u, v, k)]
        data['crime_day'] = row['crime_score_day']
        data['crime_night'] = row['crime_score_night']
        data['light'] = row['light_score']
        data['walk'] = row['walk_score']
        data['road'] = row['road_class_score']
    except KeyError:
        # If edge isn't in CSV, use neutral defaults
        data['crime_day'] = 0.5
        data['crime_night'] = 0.5
        data['light'] = 0.5
        data['walk'] = 0.5
        data['road'] = 0.5

    # Calculate and store the cost for every profile on the edge
    for p_name, p_weights in profiles.items():
        data[f'cost_{p_name}'] = calculate_composite_cost(data, p_weights)

# 4. RUN TEST ROUTES
# Get all nodes and pick two at random for testing
nodes = list(G.nodes())
random.seed(42) # For reproducible tests

source, target = None, None
print("Searching for a valid connected route...")
for _ in range(200):
    s, t = random.sample(nodes, 2)
    if nx.has_path(G, s, t):
        source, target = s, t
        break

if not source:
    print("Error: Could not find connected nodes. Check your GraphML connectivity.")
else:
    print(f"Testing route from {source} to {target}\n")
    results = []

    for profile_name in profiles.keys():
        weight_attr = f'cost_{profile_name}'
        try:
            # 1. Find the path nodes
            path = nx.shortest_path(G, source, target, weight=weight_attr)
            
            # 2. Extract edge data along the path (Fixed logic for modern OSMnx)
            path_edge_values = []
            for u, v in zip(path[:-1], path[1:]):
                # Since it's a MultiDiGraph, multiple edges might exist between u and v
                # We select the one that has the minimum cost for our current profile
                edges = G.get_edge_data(u, v)
                best_edge = min(edges.values(), key=lambda x: x.get(weight_attr, float('inf')))
                path_edge_values.append(best_edge)
            
            # 3. Calculate statistics for this route
            stats = {
                'Profile': profile_name,
                'Steps': len(path) - 1,
                'Total_Dist (m)': sum(e.get('length', 0) for e in path_edge_values),
                'Avg_Crime_Day': np.mean([e.get('crime_day', 0) for e in path_edge_values]),
                'Avg_Crime_Night': np.mean([e.get('crime_night', 0) for e in path_edge_values]),
                'Avg_Light': np.mean([e.get('light', 0) for e in path_edge_values]),
                'Avg_Walk': np.mean([e.get('walk', 0) for e in path_edge_values])
            }
            results.append(stats)
            
        except nx.NetworkXNoPath:
            print(f"No path found for {profile_name}")

    # 5. DISPLAY VALIDATION TABLE
    df_results = pd.DataFrame(results)
    print("--- VALIDATION RANKINGS ---")
    print(df_results.sort_values('Avg_Crime_Day', ascending=False).to_string(index=False))

    # Interpretation Guide:
    # 1. 'Safety_Day' should have the highest 'Avg_Crime_Day' score.
    # 2. 'Safety_Night' should have the highest 'Avg_Light' score.
    # 3. 'Standard_Shortest' should have the lowest 'Total_Dist'.

Loading GraphML file... (This may take a minute for large files)
Loading Scores CSV...
Merging scores into graph edges and calculating costs...
Searching for a valid connected route...
Testing route from 7325966541 to 49459683

--- VALIDATION RANKINGS ---
          Profile  Steps  Total_Dist (m)  Avg_Crime_Day  Avg_Crime_Night  Avg_Light  Avg_Walk
       Safety_Day    376    45551.419308       0.968198         0.981388   0.421543  0.552772
     Safety_Night    314    44316.980839       0.943817         0.977177   0.503185  0.565231
Standard_Shortest    448    39740.380116       0.879344         0.924920   0.364211  0.564419


## Testing initial scoring on sample routes to compare outputs and validate whether rankings are reasonable and interpretable.

- Standard_Shortest has lowest total distance and lowest safety scores (Day and Night)
- Safety_Night has highest Avg_light
- Safety_Day has best Safety score for day and night
- Shortest routes has less light than routes in daytime